In [1]:
# create product table
import pandas as pd
import numpy as np

np.random.seed(42)

# Create product list
products = [f"SKU{i}" for i in range(1, 101)]

categories = ["Electronics", "Clothing", "Home", "Sports", "Beauty"]

product_df = pd.DataFrame({
    "SKU Code": products,
    "Category": np.random.choice(categories, 100),
    "Cost per Unit": np.random.uniform(5, 50, 100).round(2),
    "Selling Price": np.random.uniform(10, 120, 100).round(2)
})

product_df.head()

,SKU Code,Category,Cost per Unit,Selling Price
0,SKU1,Sports,46.17,84.33
1,SKU2,Beauty,43.25,90.87
2,SKU3,Home,25.23,33.00
3,SKU4,Beauty,9.29,69.56
4,SKU5,Beauty,21.69,86.54


In [2]:
# create sales table
dates = pd.date_range(end=pd.Timestamp.today(), periods=30)

sales_data = []

for date in dates:
    for sku in products:
        qty = np.random.poisson(5)

        if qty > 0:
            sales_data.append({
                "Date": date,
                "SKU Code": sku,
                "Quantity Sold": qty
            })

sales_df = pd.DataFrame(sales_data)

sales_df.head()

,Date,SKU Code,Quantity Sold
0,2026-02-08 16:33:11.184966,SKU1,2
1,2026-02-08 16:33:11.184966,SKU2,7
2,2026-02-08 16:33:11.184966,SKU3,2
3,2026-02-08 16:33:11.184966,SKU4,7
4,2026-02-08 16:33:11.184966,SKU5,3


In [3]:
# Merging sales table with product table
sales_df = sales_df.merge(product_df, on="SKU Code")

sales_df["Revenue"] = sales_df["Quantity Sold"] * sales_df["Selling Price"]

sales_df.head()

,Date,SKU Code,Quantity Sold,Category,Cost per Unit,Selling Price,Revenue
0,2026-02-08 16:33:11.184966,SKU1,2,Sports,46.17,84.33,168.66
1,2026-02-09 16:33:11.184966,SKU1,6,Sports,46.17,84.33,505.98
2,2026-02-10 16:33:11.184966,SKU1,7,Sports,46.17,84.33,590.31
3,2026-02-11 16:33:11.184966,SKU1,6,Sports,46.17,84.33,505.98
4,2026-02-12 16:33:11.184966,SKU1,7,Sports,46.17,84.33,590.31


In [4]:
# create inventory table
inventory_df = pd.DataFrame({
    "SKU Code": products,
    "Current Stock": np.random.randint(20, 300, 100)
})

inventory_df.head()

,SKU Code,Current Stock
0,SKU1,85
1,SKU2,180
2,SKU3,86
3,SKU4,119
4,SKU5,96


In [6]:
# save the files as CSV
sales_df.to_csv("sales_data.csv", index=False)
inventory_df.to_csv("inventory_data.csv", index=False)
product_df.to_csv("product_data.csv", index=False)

In [10]:
# Load the Data

import pandas as pd
import numpy as np

sales_df = pd.read_csv("sales_data.csv")
inventory_df = pd.read_csv("inventory_data.csv")
product_df = pd.read_csv("product_data.csv")

sales_df.head()

,Date,SKU Code,Quantity Sold,Category,Cost per Unit,Selling Price,Revenue
0,2026-02-08 16:33:11.184966,SKU1,2,Sports,46.17,84.33,168.66
1,2026-02-09 16:33:11.184966,SKU1,6,Sports,46.17,84.33,505.98
2,2026-02-10 16:33:11.184966,SKU1,7,Sports,46.17,84.33,590.31
3,2026-02-11 16:33:11.184966,SKU1,6,Sports,46.17,84.33,505.98
4,2026-02-12 16:33:11.184966,SKU1,7,Sports,46.17,84.33,590.31


In [11]:
# Calculate Total Sales per SKU
sku_sales = sales_df.groupby("SKU Code").agg({
    "Quantity Sold": "sum",
    "Revenue": "sum"
}).reset_index()

sku_sales.head()

,SKU Code,Quantity Sold,Revenue
0,SKU1,143,12059.19
1,SKU10,131,5068.39
2,SKU100,155,8315.75
3,SKU11,169,20210.71
4,SKU12,177,20567.40


In [12]:
# Calculate Average Daily Sales
sku_sales["Avg Daily Sales"] = sku_sales["Quantity Sold"] / 30

sku_sales.head()

,SKU Code,Quantity Sold,Revenue,Avg Daily Sales
0,SKU1,143,12059.19,4.766667
1,SKU10,131,5068.39,4.366667
2,SKU100,155,8315.75,5.166667
3,SKU11,169,20210.71,5.633333
4,SKU12,177,20567.40,5.900000


In [13]:
# Merge Inventory Data
inventory_analysis = sku_sales.merge(
    inventory_df,
    on="SKU Code",
    how="left"
)

inventory_analysis.head()

,SKU Code,Quantity Sold,Revenue,Avg Daily Sales,Current Stock
0,SKU1,143,12059.19,4.766667,85
1,SKU10,131,5068.39,4.366667,273
2,SKU100,155,8315.75,5.166667,72
3,SKU11,169,20210.71,5.633333,36
4,SKU12,177,20567.40,5.900000,280


In [14]:
# Calculate Inventory Cover
inventory_analysis["Inventory Cover (Days)"] = (
    inventory_analysis["Current Stock"] /
    inventory_analysis["Avg Daily Sales"]
)
inventory_analysis.head()

,SKU Code,Quantity Sold,Revenue,Avg Daily Sales,Current Stock,Inventory Cover (Days)
0,SKU1,143,12059.19,4.766667,85,17.832168
1,SKU10,131,5068.39,4.366667,273,62.519084
2,SKU100,155,8315.75,5.166667,72,13.935484
3,SKU11,169,20210.71,5.633333,36,6.390533
4,SKU12,177,20567.40,5.900000,280,47.457627


In [15]:
# Add Product Info (Cost & Category)
inventory_analysis = inventory_analysis.merge(
    product_df,
    on="SKU Code",
    how="left"
)

inventory_analysis.head()

,SKU Code,Quantity Sold,Revenue,Avg Daily Sales,Current Stock,Inventory Cover (Days),Category,Cost per Unit,Selling Price
0,SKU1,143,12059.19,4.766667,85,17.832168,Sports,46.17,84.33
1,SKU10,131,5068.39,4.366667,273,62.519084,Beauty,30.26,38.69
2,SKU100,155,8315.75,5.166667,72,13.935484,Electronics,42.70,53.65
3,SKU11,169,20210.71,5.633333,36,6.390533,Sports,22.23,119.59
4,SKU12,177,20567.40,5.900000,280,47.457627,Home,48.73,116.20


In [16]:
# Calculate Margin %
inventory_analysis["Margin %"] = (
    (inventory_analysis["Selling Price"] -
     inventory_analysis["Cost per Unit"])
    / inventory_analysis["Selling Price"]
) * 100
inventory_analysis.head()

,SKU Code,Quantity Sold,Revenue,Avg Daily Sales,Current Stock,Inventory Cover (Days),Category,Cost per Unit,Selling Price,Margin %
0,SKU1,143,12059.19,4.766667,85,17.832168,Sports,46.17,84.33,45.250800
1,SKU10,131,5068.39,4.366667,273,62.519084,Beauty,30.26,38.69,21.788576
2,SKU100,155,8315.75,5.166667,72,13.935484,Electronics,42.70,53.65,20.410065
3,SKU11,169,20210.71,5.633333,36,6.390533,Sports,22.23,119.59,81.411489
4,SKU12,177,20567.40,5.900000,280,47.457627,Home,48.73,116.20,58.063683


In [17]:
# Create Stock Risk Classification
def stock_risk(days):
    
    if days < 10:
        return "Low Stock Risk"
    
    elif days < 30:
        return "Healthy Stock"
    
    else:
        return "Overstock"

inventory_analysis["Stock Status"] = inventory_analysis[
    "Inventory Cover (Days)"
].apply(stock_risk)
inventory_analysis.head()

,SKU Code,Quantity Sold,Revenue,Avg Daily Sales,Current Stock,Inventory Cover (Days),Category,Cost per Unit,Selling Price,Margin %,Stock Status
0,SKU1,143,12059.19,4.766667,85,17.832168,Sports,46.17,84.33,45.250800,Healthy Stock
1,SKU10,131,5068.39,4.366667,273,62.519084,Beauty,30.26,38.69,21.788576,Overstock
2,SKU100,155,8315.75,5.166667,72,13.935484,Electronics,42.70,53.65,20.410065,Healthy Stock
3,SKU11,169,20210.71,5.633333,36,6.390533,Sports,22.23,119.59,81.411489,Low Stock Risk
4,SKU12,177,20567.40,5.900000,280,47.457627,Home,48.73,116.20,58.063683,Overstock


In [18]:
# Save Final Dataset
inventory_analysis.to_csv("final_inventory_analysis.csv", index=False)